# 01. 데이터 수집 (v2 — team1~4 신규 데이터 통합)

기존 9개 서울 열린데이터 API 캐시 + 팀1·2·3·4가 별도로 수집한 외부 데이터를 한 곳에서 로드·정제해 후속 노트북(02~08)이 즉시 쓸 수 있도록 `data/processed/`에 산출물로 저장한다.

In [1]:
# === 라이브러리: 표준 라이브러리 + 강의 범위(numpy/pandas) ===
import os, json, time                                  # os/json/time: 환경변수·API 응답 파싱·sleep
from pathlib import Path                               # 운영체제 독립 경로
from urllib.request import urlopen                     # HTTP 호출 (강의 외 패키지 회피 — requests 안 씀)
from urllib.error import URLError, HTTPError           # 호출 실패용 예외 타입
import numpy as np                                     # L04 수치 계산
import pandas as pd                                    # L08 데이터프레임

pd.set_option('display.max_columns', 60)               # 출력 시 컬럼 잘림 방지
pd.set_option('display.width', 160)

# 디렉토리 상수 — 모든 노트북이 동일 경로 사용 (수정은 여기 한 곳만)
PROJECT_ROOT = Path('/Users/ijunsu/Documents/Documents/capston/data_analysis/trade_area_project')
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'                # 서울 열린데이터 API 캐시 (25분기×9 API)
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'    # 후속 노트북 입력
TEAM1_DIR = PROJECT_ROOT / 'data' / 'team1 이승직 (부동산, 임대료)'
TEAM2_DIR = PROJECT_ROOT / 'data' / 'team2 (한승현 - 가맹점-창업비-매출)'
TEAM3_DIR = PROJECT_ROOT / 'data' / 'team3 (행정동·미시 입지)'
TEAM4_DIR = PROJECT_ROOT / 'data' / 'team4'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)       # 없으면 생성

print('numpy', np.__version__); print('pandas', pd.__version__)
print('team1', TEAM1_DIR.exists(), '| team2', TEAM2_DIR.exists(),
      '| team3', TEAM3_DIR.exists(), '| team4', TEAM4_DIR.exists())

numpy 2.4.4
pandas 3.0.2
team1 True | team2 True | team3 True | team4 True


## 1. 기존 9개 API 캐시 점검

In [2]:
def summarize_raw(raw_dir: Path) -> pd.DataFrame:
    """data/raw/의 {api}_{quarter}.csv를 스캔해 (API × 분기) 행수 매트릭스 생성."""
    records = []
    for path in sorted(raw_dir.glob('*.csv')):         # 정렬해 결정론적 결과
        stem = path.stem                                # 확장자 제거한 파일명
        # 'API명_YYYYQ' 패턴인지 확인 (분기는 5자리 숫자 — 예 20231)
        if '_' in stem and stem.split('_')[-1].isdigit() and len(stem.split('_')[-1]) == 5:
            api = '_'.join(stem.split('_')[:-1])
            quarter = stem.split('_')[-1]
            # 헤더 제외 라인 수를 빠르게 — 메모리 안 쓰는 카운트
            try:
                n = sum(1 for _ in path.open(encoding='utf-8-sig')) - 1
            except Exception:
                n = -1
            records.append({'api': api, 'quarter': quarter, 'rows': n})
    if not records: return pd.DataFrame()
    df = pd.DataFrame(records)
    return df.pivot(index='api', columns='quarter', values='rows').fillna(0).astype(int)

api_matrix = summarize_raw(RAW_DIR)
print('서울 열린데이터 API × 분기 행수 매트릭스')
api_matrix

서울 열린데이터 API × 분기 행수 매트릭스


quarter,20201,20202,20203,20204,20211,20212,20213,20214,20221,20222,20223,20224,20231,20232,20233,20234,20241,20242,20243,20244,20251,20252,20253,20254,20261
api,,,,,,,,,,,,,,,,,,,,,,,,,
VwsmAdstrdIxQq,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000
VwsmSignguIxQq,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700
VwsmTrdarFcltyQq,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000
VwsmTrdarFlpopQq,1650,1650,1650,1650,1650,1650,1650,1650,1650,1650,1650,1649,1649,1649,1649,1648,1649,1649,1648,1649,1650,1649,1648,1648,0
VwsmTrdarIxQq,1650,1650,1650,1650,1650,1650,1650,1650,1650,1650,1650,1650,1650,1650,1650,1650,1650,1650,1650,1650,1650,1650,1650,1650,0
VwsmTrdarSelngQq,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,0
VwsmTrdarStorQq,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,0
VwsmTrdarWrcPopltnQq,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000


## 2. team1 — 부동산·임대료·공실률·마진 proxy

In [3]:
# 팀1의 5종 산출 CSV 한 번에 로드
team1_rent       = pd.read_csv(TEAM1_DIR / 'team1_rent.csv')                        # 자치구·분기·상가유형 임대료/공실률
team1_margin     = pd.read_csv(TEAM1_DIR / 'team1_margin_proxy_detail.csv')         # 자치구·분기·업종 마진 proxy
team1_gu_margin  = pd.read_csv(TEAM1_DIR / 'team1_margin_proxy_gu_summary.csv')     # 자치구 요약(랭크)
team1_brand      = pd.read_csv(TEAM1_DIR / 'brand_data.csv')                        # 가맹점 브랜드별
team1_biz        = pd.read_csv(TEAM1_DIR / 'biz_summary.csv')                       # 업종별 가맹점 평균

print('rent:', team1_rent.shape, '| margin:', team1_margin.shape,
      '| gu_margin:', team1_gu_margin.shape,
      '| brand:', team1_brand.shape, '| biz:', team1_biz.shape)
team1_rent.head(4)

rent: (900, 8) | margin: (2100, 9) | gu_margin: (25, 8) | brand: (676, 12) | biz: (7, 18)


,gu,q,store_type,rent_index_q,rent_growth_yoy,vacancy_rate,transaction_count_q,avg_transaction_price
0,종로구,20231,소규모 상가,97.736667,1.576249,4.433333,116,1.072146e+09
1,종로구,20231,중대형 상가,98.780000,-0.064074,11.133333,116,1.072146e+09
2,종로구,20231,집합 상가,97.820000,0.405440,5.400000,116,1.072146e+09
3,종로구,20232,소규모 상가,98.056667,1.799495,5.066667,142,6.321063e+09


### 2.1 자치구·분기 단위로 평균 (상가유형 3종 통합)

In [4]:
# 한 (gu, q)에 상가유형(소규모/중대형/집합) 3행이 있으므로 평균내어 1행/(gu, q)으로 압축
rent_gu_q = (team1_rent
             .groupby(['gu', 'q'], as_index=False)
             .agg(rent_index_q=('rent_index_q', 'mean'),                # 임대료지수(높을수록 비쌈)
                  rent_growth_yoy=('rent_growth_yoy', 'mean'),          # 전년동기 대비 임대료 변화율
                  vacancy_rate=('vacancy_rate', 'mean'),                # 공실률 (%)
                  transaction_count_q=('transaction_count_q', 'mean'),  # 분기 실거래 건수
                  avg_transaction_price=('avg_transaction_price', 'mean')))
rent_gu_q['q'] = rent_gu_q['q'].astype(int)                             # int 통일 — merge 시 타입 충돌 방지
print('자치구×분기 임대료 통합:', rent_gu_q.shape)
rent_gu_q.head(5)

자치구×분기 임대료 통합: (300, 7)


,gu,q,rent_index_q,rent_growth_yoy,vacancy_rate,transaction_count_q,avg_transaction_price
0,강남구,20231,97.006488,1.119078,5.317857,131.0,6.425965e+09
1,강남구,20232,97.679048,1.634178,4.882143,206.0,5.653627e+09
2,강남구,20233,98.382738,2.301039,5.147619,193.0,6.639201e+09
3,강남구,20234,98.852857,2.818582,5.725000,216.0,6.684476e+09
4,강남구,20241,99.301607,2.372115,5.858929,218.0,1.240292e+10


## 3. team2 — 가맹점·창업비 (중복 확인)

In [5]:
# team2 폴더에도 brand·biz가 있지만 team1과 동일 여부 확인
team2_brand = pd.read_csv(TEAM2_DIR / 'brand_data.csv')
team2_biz   = pd.read_csv(TEAM2_DIR / 'biz_summary.csv')
print(f'brand 동일: {team1_brand.equals(team2_brand)}')
print(f'biz   동일: {team1_biz.equals(team2_biz)}')
print('→ team1 버전 정본 채택')

brand 동일: True
biz   동일: True
→ team1 버전 정본 채택


## 4. team3 — 행정동 단위 (대용량, chunk 처리)

In [6]:
# 54MB 파일이므로 스키마만 먼저 확인
team3_cols = pd.read_csv(TEAM3_DIR / 'team3_adstrd_integrated.csv', nrows=0).columns.tolist()
print('컬럼 수:', len(team3_cols)); print('첫 25개:', team3_cols[:25])
team3_head = pd.read_csv(TEAM3_DIR / 'team3_adstrd_integrated.csv', nrows=3)
team3_head[['adstrd_code','adstrd_nm','gu','biz','q','adstrd_sales_amt','adstrd_stor_co','adstrd_flpop']].head()

컬럼 수: 85
첫 25개: ['adstrd_code', 'adstrd_nm', 'gu', 'biz', 'q', 'SVC_INDUTY_CD', 'adstrd_sales_amt', 'adstrd_sales_count', 'MDWK_SELNG_AMT', 'WKEND_SELNG_AMT', 'MON_SELNG_AMT', 'TUES_SELNG_AMT', 'WED_SELNG_AMT', 'THUR_SELNG_AMT', 'FRI_SELNG_AMT', 'SAT_SELNG_AMT', 'SUN_SELNG_AMT', 'TMZON_00_06_SELNG_AMT', 'TMZON_06_11_SELNG_AMT', 'TMZON_11_14_SELNG_AMT', 'TMZON_14_17_SELNG_AMT', 'TMZON_17_21_SELNG_AMT', 'TMZON_21_24_SELNG_AMT', 'ML_SELNG_AMT', 'FML_SELNG_AMT']


,adstrd_code,adstrd_nm,gu,biz,q,adstrd_sales_amt,adstrd_stor_co,adstrd_flpop
0,11110515,청운효자동,종로구,한식음식점,20201,2.061039e+09,74.0,3348036.0
1,11110515,청운효자동,종로구,양식음식점,20201,1.141670e+09,32.0,3348036.0
2,11110515,청운효자동,종로구,호프-간이주점,20201,2.860243e+07,13.0,3348036.0


In [7]:
# 대용량은 chunk로 — 메모리 폭발 방지 (한 번에 20만 행)
team3_iter = pd.read_csv(
    TEAM3_DIR / 'team3_adstrd_integrated.csv',
    usecols=['gu','biz','q','adstrd_sales_amt','adstrd_stor_co','adstrd_flpop','adstrd_wrc_co'],
    chunksize=200_000)

chunks = []
for ch in team3_iter:
    # chunk 단위로 우선 groupby — 다음 단계로 넘기는 데이터를 줄임
    chunks.append(ch.groupby(['gu','biz','q'], as_index=False)
                  .agg(adstrd_sales_amt=('adstrd_sales_amt','sum'),
                       adstrd_stor_co=('adstrd_stor_co','sum'),
                       adstrd_flpop=('adstrd_flpop','sum'),
                       adstrd_wrc_co=('adstrd_wrc_co','sum')))

# 모든 chunk 합친 뒤 한 번 더 group (chunk 경계에서 같은 키가 분산돼 있을 수 있음)
team3_gu = (pd.concat(chunks, ignore_index=True)
              .groupby(['gu','biz','q'], as_index=False).sum())
team3_gu.to_csv(PROCESSED_DIR / 'team3_gu_summary.csv', index=False)
print('team3 자치구 요약 →', team3_gu.shape)

team3 자치구 요약 → (5983, 7)


## 5. team4 — 외부 거시지표

In [8]:
# (1) 인구밀도 — 자치구별 최신 연도
pop_density = pd.read_csv(TEAM4_DIR / '09-20_성별연령별인구밀도-1.csv')
pop_density.columns = pop_density.columns.str.strip()
pop_density = pop_density[pop_density['자치구'] != '합계']                   # 전체 합계 행 제거

# 자치구별로 연도가 가장 큰 1행만 (가장 최신 인구밀도)
pop_density_recent = (pop_density.sort_values('연도')
                                  .groupby('자치구', as_index=False).tail(1)
                                  [['자치구','연도','인구','인구밀도','외국인']])
# 영문 컬럼명으로 정규화 — 후속 노트북에서 통일
pop_density_recent = pop_density_recent.rename(columns={'자치구':'gu','인구':'gu_population',
                                                         '인구밀도':'gu_pop_density','외국인':'gu_foreign','연도':'pop_year'})
pop_density_recent.to_csv(PROCESSED_DIR / 'team4_pop_density_gu.csv', index=False)
pop_density_recent.head(5)

,gu,pop_year,gu_population,gu_pop_density,gu_foreign
155,마포구,2020,371890.0,15592.8721,9968.0
143,동작구,2020,391220.0,23927.8287,10352.0
131,동대문구,2020,342837.0,24109.4937,14177.0
11,강남구,2020,539231.0,13651.4177,4824.0
119,도봉구,2020,325257.0,15750.9443,2104.0


In [9]:
# (2) 폐업점포수 — 멀티헤더(연도, 업종군) → long format
closure = pd.read_csv(TEAM4_DIR / '영세자영업+자치구별+폐업+점포수_20260521100026.csv', header=[0,1])
closure.columns = [f'{a}_{b}'.strip() for a, b in closure.columns]            # 멀티헤더 → 단일 컬럼명
gu_col = [c for c in closure.columns if c.startswith('자치구별')][0]
closure = closure.rename(columns={gu_col: 'gu'})
closure = closure[closure['gu'].notna() & (closure['gu'] != '서울시')]          # 전체 합계 제거

# '전체' 폐업 점포수 컬럼만 사용
total_cols = [c for c in closure.columns if c.endswith('_전체')]
closure_long = closure[['gu'] + total_cols].copy()
closure_long.columns = ['gu'] + [c.split('_')[0] for c in total_cols]          # 컬럼명을 연도만으로
closure_long = closure_long.melt(id_vars='gu', var_name='year', value_name='closure_count')
closure_long['year'] = closure_long['year'].astype(int)
closure_long['closure_count'] = pd.to_numeric(closure_long['closure_count'], errors='coerce')
closure_long.to_csv(PROCESSED_DIR / 'team4_closure_gu_year.csv', index=False)
print('폐업점포수:', closure_long.shape)

폐업점포수: (150, 3)


In [10]:
# (3) 1인가구 — 3단 헤더 처리
single_raw = pd.read_csv(TEAM4_DIR / '1인가구(연령별)_20260521094333.csv', header=[0,1,2], low_memory=False)
single_raw.columns = ['__'.join([str(s) for s in c if str(s) != 'nan']).strip() for c in single_raw.columns]

gu_col = [c for c in single_raw.columns if '자치구별(2)' in c][0]
single_raw = single_raw.rename(columns={gu_col: 'gu'})
gender_col = [c for c in single_raw.columns if '성별' in c][0]
total_rows = single_raw[single_raw[gender_col].astype(str).str.strip().isin(['계','합계'])]    # 성별=계만

year_total_cols = [c for c in single_raw.columns if '계__계' in c or c.endswith('__계')]
keep = ['gu'] + [c for c in year_total_cols if c.split('__')[0].isdigit()]
if not keep[1:]:                                                                # 매칭 실패 시 백업
    numeric_cols = [c for c in single_raw.columns if c.split('__')[0].isdigit()][:15]
    keep = ['gu'] + numeric_cols

sg = total_rows[keep].copy() if not total_rows.empty else single_raw[keep].copy()
sg = sg.melt(id_vars='gu', var_name='year_label', value_name='single_household')
sg['year'] = sg['year_label'].str.extract(r'(\d{4})').astype(float)            # 라벨에서 연도 추출
sg['single_household'] = pd.to_numeric(sg['single_household'], errors='coerce')
sg = sg.dropna(subset=['gu','year','single_household']).groupby(['gu','year'], as_index=False)['single_household'].mean()
sg.to_csv(PROCESSED_DIR / 'team4_single_household.csv', index=False)
print('1인가구:', sg.shape)

1인가구: (26, 3)


In [11]:
# (4) CPI — 헤더 복잡 → 메타만 보관
cpi_raw = pd.read_csv(TEAM4_DIR / '소비자물가지수_21095420.csv', low_memory=False)
if (cpi_raw.columns.str.contains('원자료')).sum() > 10:
    cpi_raw = pd.read_csv(TEAM4_DIR / '소비자물가지수_21095420.csv', header=[0,1,2,3], low_memory=False)
(PROCESSED_DIR / 'team4_cpi_meta.txt').write_text(
    f'columns: {cpi_raw.shape[1]}, rows: {cpi_raw.shape[0]}\n* 헤더 복잡 → 후속에서 수동 정리', encoding='utf-8')
print('CPI shape:', cpi_raw.shape, '→ 메타만 기록')

CPI shape: (181, 582) → 메타만 기록


## 6. 산출물 저장 + 요약

In [12]:
# 추가 산출물 저장
rent_gu_q.to_csv(PROCESSED_DIR / 'team1_rent_gu_q.csv', index=False)
team1_gu_margin.to_csv(PROCESSED_DIR / 'team1_margin_gu.csv', index=False)
print('processed/ 산출물:')
for p in sorted(PROCESSED_DIR.glob('*.csv')):
    print(f'  {p.name:40s} {p.stat().st_size/1024:>10.1f} KB')

processed/ 산출물:
  features.csv                                17594.9 KB
  features_adstrd.csv                          9365.6 KB
  integrated.csv                              16147.9 KB
  model_comparison.csv                            3.7 KB
  team1_margin_gu.csv                             2.7 KB
  team1_rent_gu_q.csv                            25.7 KB
  team3_gu_summary.csv                          416.5 KB
  team4_closure_gu_year.csv                       3.0 KB
  team4_pop_density_gu.csv                        1.1 KB
  team4_single_household.csv                      0.8 KB
